In [4]:
from dataclasses import dataclass
from typing import Optional
from yargy import Parser, rule, or_, and_, not_
from yargy.predicates import gram, dictionary, type, eq, normalized, gte, lte, is_title, caseless, is_capitalized, Predicate
from yargy.interpretation import fact, attribute
from yargy.relations import gnc_relation

import pymorphy3

@dataclass
class Entry:
    name: str
    birth_date: Optional[str] = None
    birth_place: Optional[str] = None

    def to_dict(self):
        return {
            "name": self.name,
            "birth_date": self.birth_date,
            "birth_place": self.birth_place,
        }

PersonName = fact("PersonName", ["first", "last", "middle"])
Date = fact("Date", ["day", "month", "year"])
Place = fact("Place", ["value"])
EntryFact = fact("EntryFact", ["person", "date", "place"])

gnc = gnc_relation()

class LengthPredicate(Predicate):
    def __init__(self, min_length):
        self.min_length = min_length
        super().__init__()

    def __call__(self, token):
        return len(token.value) >= self.min_length

class NotInSetPredicate(Predicate):
    def __init__(self, stop_words):
        self.stop_words = set(w.lower() for w in stop_words)
        super().__init__()

    def __call__(self, token):
        return token.value.lower() not in self.stop_words

# Используем
COMMON_PREPS = {"в", "из", "с", "к", "у", "о", "на", "по", "за", "под", "над", "от", "до"}

NOT_PREP = NotInSetPredicate(COMMON_PREPS)
MIN_NAME_LEN = LengthPredicate(2)
MIN_PATR_LEN = LengthPredicate(3)
def proper_name():
    return and_(
        gram("Name"),
        NOT_PREP,
        #not_(type("PNCT")),
        MIN_NAME_LEN,
        is_capitalized()
    )

def proper_surname():
    return and_(
        gram("Surn"),
        NOT_PREP,
        #not_(type("PNCT")),
        MIN_NAME_LEN,
        is_capitalized()
    )

def proper_patronymic():
    return and_(
        gram("Patr"),
        NOT_PREP,
        #not_(type("PNCT")),
        MIN_PATR_LEN,
        is_capitalized()
    )

NAME = or_(
    # ФОИ
    rule(
        proper_name().interpretation(PersonName.first),
        proper_patronymic().interpretation(PersonName.middle).optional(),
        proper_surname().interpretation(PersonName.last),
    ),
    # ФИО
    rule(
        proper_surname().interpretation(PersonName.last),
        proper_name().interpretation(PersonName.first),
        proper_patronymic().interpretation(PersonName.middle).optional(),
    ),
    # ИФ
    rule(
        proper_name().interpretation(PersonName.first),
        proper_surname().interpretation(PersonName.last).optional(),
    ),
).interpretation(PersonName)

# Дата
MONTHS = {
    "январь": 1,
    "января": 1,
    "февраль": 2,
    "февраля": 2,
    "март": 3,
    "марта": 3,
    "апрель": 4,
    "апреля": 4,
    "май": 5,
    "мая": 5,
    "июнь": 6,
    "июня": 6,
    "июль": 7,
    "июля": 7,
    "август": 8,
    "августа": 8,
    "сентябрь": 9,
    "сентября": 9,
    "октябрь": 10,
    "октября": 10,
    "ноябрь": 11,
    "ноября": 11,
    "декабрь": 12,
    "декабря": 12,
}

MONTH_WORD = dictionary(MONTHS).interpretation(Date.month.custom(MONTHS.get))

DAY = and_(type("INT"), gte(1), lte(31)).interpretation(Date.day.custom(int))
YEAR = and_(type("INT"), gte(0), lte(2026)).interpretation(Date.year.custom(int))

YEAR_SUFFIX = dictionary({"год", "года", "г.", "году", "г", "р.", "г.р.", "г.р", "р"})

FULL_DATE = rule(DAY.optional(), MONTH_WORD.optional(), YEAR, YEAR_SUFFIX.optional())


DOTTED_DATE = rule(
    type("INT").interpretation(Date.day.custom(int)),
    eq("."),
    type("INT").interpretation(Date.month.custom(int)),
    eq("."),
    type("INT").interpretation(Date.year.custom(int)),
).interpretation(Date)

DATE = or_(FULL_DATE.interpretation(Date), DOTTED_DATE)

# Место рождения
PLACE_PREP = or_(normalized("в"), normalized("из"))

# # вариант типа: Санкт-Петербург
CAPITALIZED_GEO = rule(
    and_(
        is_capitalized(),
        not_(gram("Name")),
        not_(gram("Surn")),
        not_(gram("Patr")),
        not_(type("INT")),
    ),
    eq('-').optional(),
    and_(
        is_capitalized(),
        not_(gram("Name")),
        not_(gram("Surn")),
        not_(gram("Patr")),
        not_(type("INT")),
    ).optional()
    )


PLACE = rule(PLACE_PREP, CAPITALIZED_GEO.interpretation(Place.value)).interpretation(
    Place
)


# контекст у дня рождения
BIRTH_PHRASE = or_(
    rule(normalized("родиться")),
    rule(normalized("родился")),
    rule(normalized("родилась")),
    rule(dictionary({"дата", "день"}), dictionary({"рождения", "рождение"})),
    rule(normalized("дата"), dictionary({"рождения", "рождение"})),
)

BIRTH_EVENT = rule(
    NAME.interpretation(EntryFact.person),
    BIRTH_PHRASE,
    DATE.interpretation(EntryFact.date).optional(),
    PLACE.interpretation(EntryFact.place).optional(),
)

# Вариант: "ФИО, 12.08.1978, г.р., в Москве"
BIRTH_ABBR = rule(
    NAME.interpretation(EntryFact.person),
    DATE.interpretation(EntryFact.date),
    YEAR_SUFFIX.optional(),
    PLACE.interpretation(EntryFact.place).optional(),
)


BIRTH_CONTEXT = or_(
    rule(normalized("родиться")),
    rule(normalized("родился")),
    rule(normalized("родилась")),
    rule(normalized("родился"), normalized("в")),
    rule(normalized("родилась"), normalized("в")),
    rule(dictionary({"дата", "день"}), normalized("рождения")),
    rule(caseless("г."), caseless("р.")),
    rule(caseless("д."), caseless("р.")),
)

# имя + контекст рождения + дата + место
RULE_1 = rule(
    NAME.interpretation(EntryFact.person),
    BIRTH_CONTEXT,
    DATE.interpretation(EntryFact.date).optional(),
    normalized("в").optional(),
    PLACE.interpretation(EntryFact.place).optional(),
)

# имя + дата + (контекст) + место
RULE_2 = rule(
    NAME.interpretation(EntryFact.person),
    normalized("в").optional(),
    DATE.interpretation(EntryFact.date).optional(),
    BIRTH_CONTEXT.optional(),
    PLACE.interpretation(EntryFact.place).optional(),
)

# контекст + дата + место + имя
RULE_3 = rule(
    BIRTH_CONTEXT,
    DATE.interpretation(EntryFact.date).optional(),
    normalized("в").optional(),
    PLACE.interpretation(EntryFact.place).optional(),
    NAME.interpretation(EntryFact.person),
)

# имя + место + контекст + дата
RULE_4 = rule(
    NAME.interpretation(EntryFact.person),
    normalized("в").optional(),
    PLACE.interpretation(EntryFact.place).optional(),
    BIRTH_CONTEXT.optional(),
    DATE.interpretation(EntryFact.date).optional(),
)

# имя и дата
RULE_5 = rule(
    NAME.interpretation(EntryFact.person),
    eq(',').optional(),
    DATE.interpretation(EntryFact.date),
    YEAR_SUFFIX.optional(),
)

# RULE_6 = rule(
#     NAME.interpretation(EntryFact.person)
# )


ENTRY_RULE = or_(
    RULE_1,
    RULE_2,
    RULE_3,
    RULE_4,
    RULE_5
    #RULE_6
).interpretation(EntryFact)

In [5]:
def format_name(person):
    if not person:
        return None
    parts = []
    if getattr(person, "last", None):
        parts.append(person.last)
    if getattr(person, "first", None):
        parts.append(person.first)
    if getattr(person, "middle", None):
        parts.append(person.middle)
    return " ".join(parts) if parts else None


def format_date(date):
    if not date:
        return None
    try:
        if hasattr(date, "day") and hasattr(date, "month") and hasattr(date, "year"):
            months = [
                "",
                "января",
                "февраля",
                "марта",
                "апреля",
                "мая",
                "июня",
                "июля",
                "августа",
                "сентября",
                "октября",
                "ноября",
                "декабря",
            ]
            return f"{date.day} {months[date.month]} {date.year}"
        elif hasattr(date, "year"):
            return str(date.year)
    except (TypeError, IndexError, AttributeError):
        pass
    return None

morph = pymorphy3.MorphAnalyzer()

def format_place(place):
    if place:
        #norm = morph.parse(place.value)[0].normal_form
        #if norm:
        #    return norm
        #else:
        return place.value
    return None


parser = Parser(ENTRY_RULE)

def extract_entries(text: str) -> list[Entry]:
    matches = parser.findall(text)
    results = []
    for match in matches:
        f = match.fact
        if not f or not f.person:
            continue
        name = format_name(f.person)
        birth_date = format_date(f.date)
        birth_place = format_place(f.place)
        results.append(Entry(name=name, birth_date=birth_date, birth_place=birth_place))
    return results

In [6]:
test_texts = [
    "Василий Николаев родился 16 января 1972 года в Москве. Мария Иванова родилась 5 марта 1985 года в Санкт-Петербурге.",
    "Бетси Палмер родилась в 1926 году в США в семье выходца из Чехии.",
    "Мария Иванова родилась 5 марта 1985 года в Санкт-Петербурге.",
    "Петров Сергей Викторович, дата рождения: 12.08.1978, город: Екатеринбург",
    "Родился 3 сентября 1990 года. Фамилия: Смирнов, имя: Алексей.",
    "Анна родилась в 1992 году в Казани.",
    "Николаев Василий, 16 января 1972 г.р., Москва.",
    "В 1994 году окончил университет. Родился в Новосибирске 10.10.1970.",
    "Иванов Иван — профессор, родился в Туле.",  # без даты → всё равно извлечь имя + место
    "Поляков Антон, бывший сотрудник банка, был рожден в селе Староглухово 12 января 2013 года"
]

for text in test_texts:
    entries = extract_entries(text)
    print(f"\nТекст: {text}")
    for e in entries:
        print(f" → {e}")


Текст: Василий Николаев родился 16 января 1972 года в Москве. Мария Иванова родилась 5 марта 1985 года в Санкт-Петербурге.
 → Entry(name='Николаев Василий', birth_date='16 января 1972', birth_place='Москве')
 → Entry(name='Иванова Мария', birth_date='5 марта 1985', birth_place='Санкт-Петербурге')

Текст: Бетси Палмер родилась в 1926 году в США в семье выходца из Чехии.
 → Entry(name='Бетси', birth_date=None, birth_place=None)

Текст: Мария Иванова родилась 5 марта 1985 года в Санкт-Петербурге.
 → Entry(name='Иванова Мария', birth_date='5 марта 1985', birth_place='Санкт-Петербурге')

Текст: Петров Сергей Викторович, дата рождения: 12.08.1978, город: Екатеринбург
 → Entry(name='Петров Сергей Викторович', birth_date=None, birth_place=None)

Текст: Родился 3 сентября 1990 года. Фамилия: Смирнов, имя: Алексей.
 → Entry(name='Алексей', birth_date=None, birth_place=None)

Текст: Анна родилась в 1992 году в Казани.
 → Entry(name='Анна', birth_date=None, birth_place='Казани')

Текст: Николаев 

In [8]:
import gzip
first_sentence2wath = 15
i = 0
entries = []
with gzip.open("news.txt.gz", "rt", encoding="utf-8") as f:
    for line in f:
        *_, text = line.strip().split('\t')
        entries.extend(temp:=extract_entries(text))
        if i < first_sentence2wath:
            print(temp)
        i += 1
            

[Entry(name='Джиралья', birth_date=None, birth_place=None)]
[Entry(name='Сундин Матс', birth_date=None, birth_place=None), Entry(name='Мортса', birth_date=None, birth_place=None), Entry(name='Сундин', birth_date=None, birth_place=None), Entry(name='Мортса', birth_date=None, birth_place=None), Entry(name='Сундин', birth_date=None, birth_place=None), Entry(name='Сундина', birth_date=None, birth_place=None), Entry(name='Сундин', birth_date=None, birth_place=None), Entry(name='Олимпиаде', birth_date=None, birth_place=None), Entry(name='Сундин', birth_date=None, birth_place=None), Entry(name='Юргордене', birth_date=None, birth_place=None), Entry(name='Сундина', birth_date=None, birth_place=None), Entry(name='Сундин', birth_date=None, birth_place=None)]
[Entry(name='Филиппов Владимир', birth_date=None, birth_place=None), Entry(name='Фахриеву Ильгизу', birth_date=None, birth_place=None), Entry(name='Прохоровым Михаилом', birth_date=None, birth_place=None), Entry(name='АКАР', birth_date=None, 

In [9]:
import json


entries_dict = [e.to_dict() for e in entries]
with open("entries.json", "w", encoding="utf-8") as f:
    json.dump(entries_dict, f, ensure_ascii=False, indent=2)

In [10]:
for e in entries:
    if e.birth_date or e.birth_place:
        print(e)

Entry(name='Тоёде', birth_date=None, birth_place='Toyota')
Entry(name='Труве', birth_date=None, birth_place='США')
Entry(name='Олимпиады', birth_date=None, birth_place='Сочи')
Entry(name='Эйлера Леонарда', birth_date=None, birth_place='XVIII')
Entry(name='Олимпиады', birth_date=None, birth_place='Сочи')
Entry(name='Родена', birth_date=None, birth_place='Париже')
Entry(name='Каррара', birth_date=None, birth_place='Бергамо')
Entry(name='Саркози Николя', birth_date=None, birth_place='Facebook')
Entry(name='Смит', birth_date=None, birth_place='Red Hot')
Entry(name='Алису', birth_date=None, birth_place='Стране Чудес')
Entry(name='Олимпиаде', birth_date=None, birth_place='Сочи')
Entry(name='Олимпиады', birth_date=None, birth_place='Сочи')
Entry(name='Олимпиада', birth_date=None, birth_place='Лондоне')
Entry(name='Олимпиады', birth_date=None, birth_place='Сочи')
Entry(name='Джонсом', birth_date=None, birth_place='Панамы')
Entry(name='Поля', birth_date=None, birth_place='Чикаго')
Entry(name='О